# Project 8: hoe snel kwam Suriname online?

**De vraag.** Hoe snel begonnen mensen in Suriname tussen 1995 en 2024 internet en mobiele telefoons te gebruiken? In welk jaar gebruikte voor het eerst meer dan de helft van de bevolking internet, en hoe snel groeide dat aandeel?

**Voor wie het is.** Een ambtenaar van een ministerie dat een openbare dienst, zoals een aanvraagformulier, online wil brengen. De ambtenaar moet weten hoeveel mensen die dienst vandaag kunnen bereiken en wat de data niet kan zeggen.

**De data.** Twee reeksen van de Wereldbank voor Suriname, allebei verzameld door de Internationale Telecommunicatie-unie (ITU). Indicator IT.NET.USER.ZS telt de mensen die in de afgelopen drie maanden internet gebruikten, als percentage van de bevolking. Indicator IT.CEL.SETS.P2 telt mobiele abonnementen per 100 inwoners. De Wereldbank heeft beide reeksen voor het laatst bijgewerkt op 2026-10-08. De tabel heeft één rij per reeks per jaar.

**Je team.** Bouwer, datacontroleur, AI-controleur en presentator. Deze week commit elke rol ook zijn eigen werk in de repository van het team, zodat de geschiedenis laat zien wie wat deed.

**Wat je vrijdag inlevert.** Het adres van de repository van je team op GitHub. Daarin staan dit notebook met elke CHECK geslaagd, het databestand, één grafiek, een README die een buitenstaander kan volgen, een notitie van één pagina voor de ambtenaar van het ministerie, en jullie AI-logboek.

## Laad de data

Deze cel krijg je cadeau. Hij leest de tabel uit de cursusmap, of rechtstreeks bij de Wereldbank als het bestand er niet is.

In [ ]:
import json, urllib.request
import pandas as pd

# IT.NET.USER.ZS: personen die internet gebruiken, % van de bevolking
# IT.CEL.SETS.P2: mobiele abonnementen per 100 inwoners
URL = ("https://api.worldbank.org/v2/country/SUR/indicator/{code}"
       "?format=json&date=1995:2024&per_page=100")
try:
    online = pd.read_csv("https://santerre.ai/bigi-data/data/week08-online.csv")
except FileNotFoundError:
    rows = []
    for code in ["IT.NET.USER.ZS", "IT.CEL.SETS.P2"]:
        raw = json.load(urllib.request.urlopen(URL.format(code=code)))[1]
        rows += [{"indicator": code, "year": int(r["date"]), "value": r["value"]} for r in raw]
    online = pd.DataFrame(rows).sort_values(["indicator", "year"]).reset_index(drop=True)
online

## TODO 1: beschrijf de tabel

Druk het aantal rijen, de twee indicatoren en het eerste en laatste jaar af. Sla in `missing` het aantal ontbrekende waarden per indicator op, als Series met de indicatorcode als index.

In [ ]:
# TODO 1
missing = None

In [ ]:
# CHECK 1
expected = online.groupby("indicator")["value"].apply(lambda s: int(s.isna().sum()))
assert missing is not None, "missing is nog leeg"
assert {k: int(v) for k, v in dict(missing).items()} == expected.to_dict()
print("CHECK 1 geslaagd:", expected.to_dict())

## TODO 2: één rij per jaar

Maak een tabel `wide` met één rij per jaar, het jaar als index, en twee kolommen: `internet` voor IT.NET.USER.ZS en `mobile` voor IT.CEL.SETS.P2. Zoek `pivot` op in pandas, of vraag je AI-assistent.

In [ ]:
# TODO 2
wide = None

In [ ]:
# CHECK 2
assert wide is not None, "wide is nog leeg"
assert list(wide.columns) == ["internet", "mobile"], list(wide.columns)
assert sorted(wide.index) == sorted(online["year"].unique()), "gebruik één rij per jaar, met het jaar als index"
assert wide["internet"].isna().sum() == expected["IT.NET.USER.ZS"]
print("CHECK 2 geslaagd:", wide.shape[0], "jaren en", wide.shape[1], "kolommen")

## TODO 3: maak een grafiek van beide reeksen

Zet beide reeksen tegen het jaar uit in één grafiek, met markeringen, een titel, benoemde assen, een legenda en de bron. Laat de gaten staan: teken `wide` zoals hij is, zonder eerst de ontbrekende jaren weg te laten. In Colab bewaart `fig.savefig("suriname-online.png", dpi=150)` de grafiek als bestand voor je repository.

In [ ]:
# TODO 3

## TODO 4: voorbij de helft

Sla in `first_internet_50` het eerste jaar op waarin meer dan 50 procent van de mensen internet gebruikte. Sla in `first_mobile_50` het eerste jaar op met meer dan 50 mobiele abonnementen per 100 inwoners.

Een abonnement telt een account, en één persoon kan er meerdere hebben. De Wereldbank telt abonnementen achteraf en prepaidaccounts die in de afgelopen drie maanden zijn gebruikt. De twee getallen meten verschillende dingen, dus schrijf dat op voordat je ze vergelijkt.

In [ ]:
# TODO 4
first_internet_50 = None
first_mobile_50 = None

In [ ]:
# CHECK 4
assert first_internet_50 is not None and first_mobile_50 is not None, "beide jaren zijn nog leeg"
assert int(first_internet_50) == wide.index[wide["internet"] > 50].min()
assert int(first_mobile_50) == wide.index[wide["mobile"] > 50].min()
print("CHECK 4 geslaagd: internet", int(first_internet_50), "en mobiel", int(first_mobile_50))

## TODO 5: hoe snel, en waar de gegevens ophouden

Sla in `gap_years` een lijst op van de jaren waarin de internetreeks geen waarde heeft. Sla in `pace_2010_2019` de gemiddelde groei in procentpunten per jaar op van 2010 tot 2019. Sla in `pace_across_gap` de gemiddelde groei per jaar op van 2019 tot het eerstvolgende jaar met een waarde.

Het tweede tempo is een gemiddelde over jaren die de tabel niet dekt. Het kan je niet vertellen hoe de groei over die jaren verdeeld was, en je README moet dat zeggen.

In [ ]:
# TODO 5
gap_years = None
pace_2010_2019 = None
pace_across_gap = None

In [ ]:
# CHECK 5
i = wide["internet"]
assert None not in (gap_years, pace_2010_2019, pace_across_gap), "een variabele is nog leeg"
assert sorted(int(y) for y in gap_years) == [int(y) for y in i.index[i.isna()]]
assert abs(pace_2010_2019 - (i[2019] - i[2010]) / 9) < 0.01
nxt = i.dropna().index[i.dropna().index > 2019].min()
assert abs(pace_across_gap - (i[nxt] - i[2019]) / (nxt - 2019)) < 0.01
print("CHECK 5 geslaagd:", round(pace_2010_2019, 2), "en daarna", round(pace_across_gap, 2), "punten per jaar")

## TODO 6: de grootste veranderingen in één jaar

Sla in `internet_jump_year` het jaar op met de grootste stijging van het internetgebruik in één jaar. Sla in `mobile_fall_year` het jaar op met de grootste daling van de mobiele abonnementen in één jaar.

Vergelijk elk jaar alleen met het jaar direct ervoor. Een verandering over een gat van meerdere jaren is geen verandering in één jaar. De mobiele reeks daalt in één jaar sterk, en de tabel legt die daling vast zonder de oorzaak. Zoek een bron die de oorzaak noemt voordat je er een in de README zet, of schrijf dat de oorzaak onbekend is.

In [ ]:
# TODO 6
internet_jump_year = None
mobile_fall_year = None

In [ ]:
# CHECK 6
step = wide.diff()
assert internet_jump_year is not None and mobile_fall_year is not None, "beide jaren zijn nog leeg"
assert int(internet_jump_year) == step["internet"].idxmax(), "vergelijk elk jaar alleen met het jaar direct ervoor"
assert int(mobile_fall_year) == step["mobile"].idxmin()
print("CHECK 6 geslaagd: internet", int(internet_jump_year), "en mobiel", int(mobile_fall_year))

## TODO 7: toets de AI aan de tabel

Vraag je AI-assistent, zonder hem deze tabel te laten zien: *Welk deel van de mensen in Suriname gebruikte in 2021 internet? In welk jaar gebruikte voor het eerst meer dan de helft van de Surinamers internet?* Schrijf beide antwoorden in jullie AI-logboek.

Sla daarna in `table_2021` de waarde op die deze tabel voor het internetgebruik in 2021 heeft, en vergelijk die met het antwoord van de assistent. Als de assistent een getal gaf dat niet in de tabel staat, vraag dan naar de bron, open die bron en noteer of daar staat wat de assistent beweerde.

In [ ]:
# TODO 7
table_2021 = "nog niet opgezocht"

In [ ]:
# CHECK 7
expected_2021 = wide.loc[2021, "internet"]
assert not isinstance(table_2021, str), "table_2021 is nog leeg"
assert (pd.isna(expected_2021) and pd.isna(table_2021)) or abs(table_2021 - expected_2021) < 1e-9
print("CHECK 7 geslaagd:", "de tabel heeft geen waarde voor 2021" if pd.isna(expected_2021) else f"de tabel zegt {expected_2021}")

## TODO 8: de README

Schrijf de tekst van je README in `readme`, in Markdown. Begin met een titelregel en gebruik daarna minstens vier secties: de vraag, de data (met het adres van de Wereldbank en de downloaddatum), hoe je het notebook uitvoert, wat er in de repository zit, en wat jullie vonden.

Noem je antwoorden op opdracht 4 en zeg wat de jaren zonder waarde betekenen. De CHECK-cel controleert ook dat de README geen wachtwoord of token bevat.

In [ ]:
# TODO 8
readme = None

In [ ]:
# CHECK 8
import re
assert readme is not None, "readme is nog leeg"
assert readme.lstrip().startswith("# "), "begin met een titelregel die start met # "
assert readme.count("\n## ") >= 4, "gebruik minstens vier secties die beginnen met ## "
assert "worldbank.org" in readme, "verwijs naar de bron bij de Wereldbank"
assert str(first_internet_50) in readme and str(first_mobile_50) in readme, "noem je antwoorden op opdracht 4"
assert len(readme.split()) >= 150, "schrijf minstens 150 woorden"
assert not re.search(r"ghp_|github_pat_|password\s*=|token\s*=", readme, re.I), "een README bevat nooit een wachtwoord of token"
print("CHECK 8 geslaagd:", len(readme.split()), "woorden in", readme.count("\n## "), "secties")

## TODO 9: de notitie voor het ministerie

Schrijf in een tekstcel hieronder een notitie van ongeveer 200 woorden voor een ambtenaar van een ministerie dat een openbare dienst online wil brengen. Zeg welk deel van de mensen internet gebruikte in het laatste jaar dat de tabel dekt, hoe snel dat aandeel groeide, en wat de ambtenaar niet uit de mobiele reeks moet afleiden. Noem de bron, en noem één ding dat de data niet kan vertellen, zoals wie nog offline is of waar die mensen wonen. Bewaar de notitie als `NOTE.md` in de repository.

## TODO 10: zet het werk in de repository

1. De bouwer maakt de repository op GitHub, voegt de andere drie toe als collaborators, en commit vóór al het andere een `.gitignore`.
2. Elk lid commit zijn eigen deel met een onderwerpregel die zegt wat het doet. De bouwer commit dit notebook, de datacontroleur de CSV, de AI-controleur `AI-LOG.md`, en de presentator `README.md`, `NOTE.md` en de grafiek.
3. Zodra de CSV naast het notebook staat, verander je het pad in de laadcel in `week08-online.csv`, en commit je die wijziging met een onderwerpregel die dat zegt.
4. Lees vóór elke push `git status`. Pull voordat je pusht, en forceer nooit een push.
5. Lees vóór vrijdag de hele geschiedenis één keer door en controleer dat er geen wachtwoord, token of groot bestand in voorkomt.